# Steam Indie Game — 층화 표본 추출
## 목적
`steam_indie_list.csv` 에서 **초기 리뷰 → 흥행 패턴 분석**을 위한 대표 표본 150개를 추출한다.

### 층화 기준
| 축 | 기준 | 설명 |
|----|------|------|
| 흥행 규모 | `owners_lower` | small / mid / large 3단계 |
| 리뷰 신뢰도 | Wilson Score ±% | low / mid / high 3단계 |
| F2P 비율 보정 | 모집단 비율(7.5%) 반영 | 과대표집 보정 |

### 공통 필터 조건
- `total_reviews >= 10` : 리뷰 최소 10개 이상
- `release_date >= 2017` : Steam 리뷰 정책 안정화 이후
- `type == 'game'` : 소프트웨어 / DLC 제외


In [39]:
import pandas as pd
import numpy as np
import ast
from scipy import stats
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print("라이브러리 로드 완료")


라이브러리 로드 완료


## 1. 데이터 로드 및 기본 확인

In [40]:
df = pd.read_csv("../../../data/raw/steam_indie_list.csv")

print(f"shape : {df.shape}")
print(f"columns : {df.columns.tolist()}")
print()
print(df.dtypes)
print()
df.head(3)


shape : (61266, 12)
columns : ['appid', 'spy_name', 'owners', 'positive', 'negative', 'price_spy', 'ccu', 'name_store', 'type', 'genres', 'release_date', 'developers']

appid           int64
spy_name          str
owners            str
positive        int64
negative        int64
price_spy       int64
ccu             int64
name_store        str
type              str
genres            str
release_date      str
developers        str
dtype: object



,appid,spy_name,owners,positive,negative,price_spy,ccu,name_store,type,genres,release_date,developers
0,1623730,Palworld,"50,000,000 .. 100,000,000",358266,22443,2999,18028,Palworld,game,"['Action', 'Adventure', 'Indie', 'RPG', 'Early...","18 Jan, 2024",Pocketpair
1,304930,Unturned,"50,000,000 .. 100,000,000",506516,48852,0,10408,Unturned,game,"['Action', 'Adventure', 'Casual', 'Indie', 'Fr...","7 Jul, 2017",Smartly Dressed Games
2,105600,Terraria,"20,000,000 .. 50,000,000",1373979,35494,999,24580,Terraria,game,"['Action', 'Adventure', 'Indie', 'RPG']","16 May, 2011",Re-Logic


## 2. 전처리

In [41]:
# 리뷰 수 합산
df['total_reviews'] = df['positive'] + df['negative']

# 출시일 파싱
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')

# owners 하한값 파싱 (예: "20,000 .. 50,000" → 20000)
def parse_owners_lower(s):
    try:
        return int(s.split('..')[0].strip().replace(',', ''))
    except:
        return 0

df['owners_lower'] = df['owners'].apply(parse_owners_lower)

# 장르 리스트 파싱
def parse_genres(g):
    try:
        return ast.literal_eval(g)
    except:
        return []

df['genres_list'] = df['genres'].apply(parse_genres)
df['is_f2p'] = df['genres_list'].apply(lambda gl: 'Free To Play' in gl)

print("전처리 완료")
print(f"total_reviews 기초통계:\n{df['total_reviews'].describe()}")


전처리 완료
total_reviews 기초통계:
count    6.126600e+04
mean     9.424468e+02
std      1.509816e+04
min      0.000000e+00
25%      6.000000e+00
50%      2.300000e+01
75%      1.040000e+02
max      1.409473e+06
Name: total_reviews, dtype: float64


## 3. 기본 필터 적용

In [42]:
df_f = df[
    (df['total_reviews'] >= 10) &
    (df['release_date'].dt.year >= 2017) &
    (df['type'] == 'game')
].copy()

print(f"필터 전: {len(df):,}개")
print(f"필터 후: {len(df_f):,}개")
print(f"제외됨 : {len(df) - len(df_f):,}개")


필터 전: 61,266개
필터 후: 35,479개
제외됨 : 25,787개


## 4. Wilson Score 기반 신뢰도 경계값 계산

리뷰 수가 몇 개 이상이어야 긍정 비율을 신뢰할 수 있는지를  
Wilson Score 신뢰구간 폭으로 정량화한다.

- **저신뢰 → 중신뢰 경계** : 신뢰구간 폭 ±15% 이하가 되는 리뷰 수
- **중신뢰 → 고신뢰 경계** : 신뢰구간 폭 ±5% 이하가 되는 리뷰 수


In [43]:
Z = 1.96  # 95% 신뢰수준

def wilson_margin(n, z=Z):
    """리뷰 수 n일 때 Wilson Score 신뢰구간 폭 (±%) — p=0.5 최악의 경우 기준"""
    p = 0.5
    denom = 1 + z**2 / n
    margin = (z / denom) * np.sqrt(p*(1-p)/n + z**2/(4*n**2))
    return margin * 100

def find_n_for_margin(target_pct, z=Z):
    """목표 신뢰구간 폭(±%) 달성에 필요한 최소 리뷰 수"""
    for n in range(1, 10000):
        if wilson_margin(n, z) <= target_pct:
            return n
    return 10000

LOW_MARGIN  = 15   # 저신뢰 상한 기준 (±15%)
HIGH_MARGIN = 5    # 고신뢰 하한 기준 (±5%)

LOW_BOUNDARY  = find_n_for_margin(LOW_MARGIN)   # 저→중 경계
HIGH_BOUNDARY = find_n_for_margin(HIGH_MARGIN)  # 중→고 경계

print(f"저신뢰 (low) : 리뷰 10 ~ {LOW_BOUNDARY-1}개  (신뢰구간 ±{LOW_MARGIN}% 초과)")
print(f"중신뢰 (mid) : 리뷰 {LOW_BOUNDARY} ~ {HIGH_BOUNDARY-1}개  (±{HIGH_MARGIN}~{LOW_MARGIN}%)")
print(f"고신뢰 (high): 리뷰 {HIGH_BOUNDARY}개 이상  (±{HIGH_MARGIN}% 이하)")

# 구간별 확인표
rows = []
for n in [10, 20, 38, 39, 100, 200, 380, 381, 500, 1000]:
    m = wilson_margin(n)
    tier = 'low' if n < LOW_BOUNDARY else ('mid' if n < HIGH_BOUNDARY else 'high')
    rows.append({'리뷰 수': n, '신뢰구간 폭(±%)': round(m, 1), '신뢰도 층': tier})

pd.DataFrame(rows)


저신뢰 (low) : 리뷰 10 ~ 38개  (신뢰구간 ±15% 초과)
중신뢰 (mid) : 리뷰 39 ~ 380개  (±5~15%)
고신뢰 (high): 리뷰 381개 이상  (±5% 이하)


,리뷰 수,신뢰구간 폭(±%),신뢰도 층
0,10,26.3,low
1,20,20.1,low
2,38,15.2,low
3,39,15.0,mid
4,100,9.6,mid
5,200,6.9,mid
6,380,5.0,mid
7,381,5.0,high
8,500,4.4,high
9,1000,3.1,high


## 5. 층 할당

In [44]:
def assign_stratum(row):
    # 흥행 규모 (owners 하한 기준)
    if row['owners_lower'] >= 200_000:
        scale = 'large'
    elif row['owners_lower'] >= 20_000:
        scale = 'mid'
    else:
        scale = 'small'

    # 리뷰 신뢰도 (Wilson Score 경계 기준)
    if row['total_reviews'] >= HIGH_BOUNDARY:
        trust = 'high'
    elif row['total_reviews'] >= LOW_BOUNDARY:
        trust = 'mid'
    else:
        trust = 'low'

    return f'{scale}_{trust}'

df_f['stratum'] = df_f.apply(assign_stratum, axis=1)

print("=== 층별 모집단 크기 ===")
stratum_counts = df_f['stratum'].value_counts().sort_index()
print(stratum_counts)
print(f"\n총합: {stratum_counts.sum():,}개")


=== 층별 모집단 크기 ===
stratum
large_high     1726
large_low        57
large_mid        89
mid_high       3419
mid_low        2029
mid_mid        4183
small_high      490
small_low     14278
small_mid      9208
Name: count, dtype: int64

총합: 35,479개


## 6. 층화 추출

- 분석 핵심층(`large_high`, `mid_high`)을 과대표집
- `small_low` 는 신호가 약해 최소화
- 총 150개 추출


In [45]:
SAMPLE_PLAN = {
    'large_high': 30,
    'large_mid' : 10,
    'large_low' :  5,
    'mid_high'  : 25,
    'mid_mid'   : 20,
    'mid_low'   : 10,
    'small_high': 20,
    'small_mid' : 15,
    'small_low' : 15,
}

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

sampled_frames = []

print(f"{'층':<15} {'모집단':>7}  {'추출':>5}  {'추출률':>7}")
print('-' * 40)

for stratum, n in SAMPLE_PLAN.items():
    pool     = df_f[df_f['stratum'] == stratum]
    actual_n = min(n, len(pool))
    sample   = pool.sample(n=actual_n, random_state=RANDOM_SEED)
    sampled_frames.append(sample)
    rate = actual_n / len(pool) * 100 if len(pool) > 0 else 0
    print(f"{stratum:<15} {len(pool):>7}  {actual_n:>5}  {rate:>6.1f}%")

df_sample = pd.concat(sampled_frames).reset_index(drop=True)

print('-' * 40)
print(f"{'총계':<15} {len(df_f):>7}  {len(df_sample):>5}")


층                   모집단     추출      추출률
----------------------------------------
large_high         1726     30     1.7%
large_mid            89     10    11.2%
large_low            57      5     8.8%
mid_high           3419     25     0.7%
mid_mid            4183     20     0.5%
mid_low            2029     10     0.5%
small_high          490     20     4.1%
small_mid          9208     15     0.2%
small_low         14278     15     0.1%
----------------------------------------
총계                35479    150


## 7. F2P 비율 보정

모집단 F2P 비율(7.5%)에 맞게 표본 내 과대표집된 F2P를 줄이고  
비F2P 게임으로 보충한다.


In [46]:
F2P_POPULATION_RATE = df_f['is_f2p'].mean()
F2P_TARGET = round(len(df_sample) * F2P_POPULATION_RATE)

current_f2p = df_sample['is_f2p'].sum()
print(f"모집단 F2P 비율 : {F2P_POPULATION_RATE:.1%}")
print(f"조정 전 F2P 수  : {current_f2p}개 ({current_f2p/len(df_sample):.1%})")
print(f"목표 F2P 수     : {F2P_TARGET}개 ({F2P_TARGET/len(df_sample):.1%})")

excess = current_f2p - F2P_TARGET

if excess > 0:
    # F2P 게임 중 excess개 제거
    f2p_idx  = df_sample[df_sample['is_f2p']].index.tolist()
    np.random.seed(RANDOM_SEED)
    drop_idx = np.random.choice(f2p_idx, size=excess, replace=False)
    df_sample = df_sample.drop(index=drop_idx).reset_index(drop=True)

    # 제거한 수만큼 비F2P로 보충 (이미 추출된 appid 제외, 층 분포 유지)
    used_appids = set(df_sample['appid'])
    non_f2p_pool = df_f[(~df_f['is_f2p']) & (~df_f['appid'].isin(used_appids))]
    fill_samples = []
    current_counts = df_sample['stratum'].value_counts().to_dict()

    for i in range(excess):
        # 모집단 대비 추출률이 낮은 층 우선 보충
        best_stratum = max(
            SAMPLE_PLAN.keys(),
            key=lambda s: len(df_f[df_f['stratum'] == s]) - current_counts.get(s, 0)
        )
        pool_fill = non_f2p_pool[
            (non_f2p_pool['stratum'] == best_stratum) &
            (~non_f2p_pool['appid'].isin(used_appids))
        ]
        if len(pool_fill) > 0:
            picked = pool_fill.sample(1, random_state=RANDOM_SEED + i)
            fill_samples.append(picked)
            used_appids.add(picked['appid'].values[0])
            current_counts[best_stratum] = current_counts.get(best_stratum, 0) + 1

    if fill_samples:
        df_sample = pd.concat([df_sample] + fill_samples).reset_index(drop=True)

    final_f2p = df_sample['is_f2p'].sum()
    print(f"\n조정 후 F2P 수  : {final_f2p}개 ({final_f2p/len(df_sample):.1%})")
    print(f"최종 표본 수    : {len(df_sample)}개")
else:
    print("\nF2P 과대표집 없음 — 조정 불필요")


모집단 F2P 비율 : 7.5%
조정 전 F2P 수  : 25개 (16.7%)
목표 F2P 수     : 11개 (7.3%)

조정 후 F2P 수  : 11개 (7.3%)
최종 표본 수    : 150개


## 8. 표본 검증

In [47]:
print("=== 층별 최종 구성 ===")
print(df_sample['stratum'].value_counts().sort_index())

print("\n=== 출시연도 분포 ===")
print(df_sample['release_date'].dt.year.value_counts().sort_index())

print("\n=== 장르 분포 (표본 vs 모집단) ===")
sample_genre_counter = Counter()
pop_genre_counter    = Counter()

for gl in df_sample['genres_list']:
    for g in gl:
        sample_genre_counter[g] += 1

for gl in df_f['genres_list']:
    for g in gl:
        pop_genre_counter[g] += 1

rows = []
for genre, _ in pop_genre_counter.most_common(10):
    s_pct = sample_genre_counter[genre] / len(df_sample) * 100
    p_pct = pop_genre_counter[genre]    / len(df_f)     * 100
    rows.append({
        '장르': genre,
        '표본(%)': round(s_pct, 1),
        '모집단(%)': round(p_pct, 1),
        '차이(%)': round(s_pct - p_pct, 1)
    })

pd.DataFrame(rows)


=== 층별 최종 구성 ===
stratum
large_high    25
large_low      5
large_mid     10
mid_high      23
mid_low       10
mid_mid       15
small_high    20
small_low     28
small_mid     14
Name: count, dtype: int64

=== 출시연도 분포 ===
release_date
2017    17
2018    18
2019    21
2020    19
2021    17
2022    15
2023    14
2024    18
2025    11
Name: count, dtype: int64

=== 장르 분포 (표본 vs 모집단) ===


,장르,표본(%),모집단(%),차이(%)
0,Indie,100.0,99.9,0.1
1,Adventure,50.7,46.6,4.1
2,Action,40.7,44.4,-3.7
3,Casual,39.3,43.6,-4.2
4,Simulation,20.7,23.7,-3.1
5,Strategy,22.0,20.7,1.3
6,RPG,20.7,20.5,0.2
7,Early Access,8.0,9.6,-1.6
8,Free To Play,7.3,7.5,-0.2
9,Sports,4.7,4.2,0.4


## 9. 저장

In [48]:
OUTPUT_PATH = "../../../data/processed/steam_stratified_sample.csv"

out_cols = [
    'appid', 'name_store', 'release_date', 'genres',
    'owners', 'owners_lower', 'positive', 'negative',
    'total_reviews', 'price_spy', 'ccu', 'developers',
    'stratum', 'is_f2p'
]

df_sample[out_cols].to_csv(OUTPUT_PATH, index=False)
print(f"저장 완료 → {OUTPUT_PATH}  ({len(df_sample)}개)")
df_sample[out_cols].head()


저장 완료 → ../../../data/processed/steam_stratified_sample.csv  (150개)


,appid,name_store,release_date,genres,owners,owners_lower,positive,negative,total_reviews,price_spy,ccu,developers,stratum,is_f2p
0,1392820,Milk inside a bag of milk inside a bag of milk,2020-08-26,"['Adventure', 'Indie']","500,000 .. 1,000,000",500000,28937,998,29935,149,13,Nikita Kryukov,large_high,False
1,1294420,Rollerdrome,2022-08-16,"['Action', 'Indie', 'Sports']","200,000 .. 500,000",200000,1891,198,2089,749,2,Roll7,large_high,False
2,537340,Guts and Glory,2018-07-19,"['Action', 'Casual', 'Indie', 'Racing']","200,000 .. 500,000",200000,2127,534,2661,1499,3,HakJak,large_high,False
3,1059990,Trombone Champ,2022-09-15,"['Casual', 'Indie']","200,000 .. 500,000",200000,9577,200,9777,1499,33,Holy Wow Studios LLC,large_high,False
4,615530,"Love, Money, Rock'n'Roll",2022-08-04,"['Adventure', 'Casual', 'Indie']","200,000 .. 500,000",200000,5504,1369,6873,499,19,Soviet Games,large_high,False


## 10. 추출 표본 분포 확인 (박스플롯)

In [49]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

TARGET_STRATA_VIS = ['large_high', 'mid_high']
STRATUM_POS       = {'large_high': 1, 'mid_high': 2}
COLORS            = {'large_high': '#4C72B0', 'mid_high': '#DD8452'}

df_f['positive_rate']      = df_f['positive'] / df_f['total_reviews']
df_target = df_sample[df_sample['stratum'].isin(TARGET_STRATA_VIS)].copy()
df_target['positive_rate'] = df_target['positive'] / df_target['total_reviews']

metrics = [
    ('total_reviews',  '총 리뷰 수',     True),
    ('owners_lower',   '소유자 수 하한', True),
    ('positive_rate',  '긍정 리뷰 비율', False),
]

fig = make_subplots(rows=1, cols=3,
                    subplot_titles=[m[1] for m in metrics],
                    horizontal_spacing=0.08)

np.random.seed(42)
legend_keys = set()

for col_idx, (col, label, log_scale) in enumerate(metrics, start=1):
    val_fmt = '{:.1%}' if col == 'positive_rate' else '{:,.0f}'

    for stratum in TARGET_STRATA_VIS:
        pos   = STRATUM_POS[stratum]
        color = COLORS[stratum]
        r, g, b = int(color[1:3], 16), int(color[3:5], 16), int(color[5:7], 16)

        pop_vals  = df_f[df_f['stratum'] == stratum][col].dropna().values
        samp_rows = df_target[df_target['stratum'] == stratum].dropna(subset=[col]).reset_index(drop=True)
        samp_vals = samp_rows[col].values

        q1, q3  = np.percentile(pop_vals, 25), np.percentile(pop_vals, 75)
        in_iqr  = (samp_vals >= q1) & (samp_vals <= q3)
        jitter  = np.random.uniform(-0.12, 0.12, size=len(samp_vals))

        # ── 모집단 박스플롯 ──
        box_key = f'box_{stratum}'
        fig.add_trace(go.Box(
            x=np.full(len(pop_vals), pos),
            y=pop_vals,
            name=stratum,
            legendgroup=box_key,
            showlegend=box_key not in legend_keys,
            marker_color=color,
            fillcolor=f'rgba({r},{g},{b},0.25)',
            line_color=color,
            boxpoints=False,
            width=0.3,
        ), row=1, col=col_idx)
        legend_keys.add(box_key)

        # ── 표본: IQR 내 (속이 찬 원) ──
        in_key  = f'{stratum}_in'
        mask_in = in_iqr
        if mask_in.any():
            hover = [
                f"{row['name_store']}<br>{label}: {val_fmt.format(v)}<br>● IQR 내"
                for v, (_, row) in zip(samp_vals[mask_in], samp_rows[mask_in].iterrows())
            ]
            fig.add_trace(go.Scatter(
                x=pos + jitter[mask_in],
                y=samp_vals[mask_in],
                mode='markers',
                name=f'{stratum} 표본 IQR 내',
                legendgroup=in_key,
                showlegend=in_key not in legend_keys,
                marker=dict(color=color, size=9, line=dict(color='white', width=1)),
                text=hover,
                hovertemplate='%{text}<extra></extra>',
            ), row=1, col=col_idx)
            legend_keys.add(in_key)

        # ── 표본: IQR 외 (속이 빈 원) ──
        out_key  = f'{stratum}_out'
        mask_out = ~in_iqr
        if mask_out.any():
            hover = [
                f"{row['name_store']}<br>{label}: {val_fmt.format(v)}<br>○ IQR 외"
                for v, (_, row) in zip(samp_vals[mask_out], samp_rows[mask_out].iterrows())
            ]
            fig.add_trace(go.Scatter(
                x=pos + jitter[mask_out],
                y=samp_vals[mask_out],
                mode='markers',
                name=f'{stratum} 표본 IQR 외',
                legendgroup=out_key,
                showlegend=out_key not in legend_keys,
                marker=dict(color='white', size=9, line=dict(color=color, width=2)),
                text=hover,
                hovertemplate='%{text}<extra></extra>',
            ), row=1, col=col_idx)
            legend_keys.add(out_key)

    fig.update_xaxes(
        tickmode='array',
        tickvals=[1, 2],
        ticktext=TARGET_STRATA_VIS,
        range=[0.5, 2.5],
        row=1, col=col_idx,
    )
    if log_scale:
        fig.update_yaxes(type='log', row=1, col=col_idx)

fig.update_layout(
    title=dict(
        text=f'모집단 분포 대비 추출 표본 위치 (n={len(df_target)}개)<br>'
             f'<sup>● 속이 찬 점 = IQR 내  ○ 속이 빈 점 = IQR 외</sup>',
        font_size=14,
    ),
    height=580,
    plot_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=-0.28,
                xanchor='center', x=0.5, font_size=11),
)
fig.update_yaxes(gridcolor='#eeeeee', gridwidth=1)

fig.show()

# IQR 포함 비율 요약
print("\n=== 추출 표본 중 IQR(Q1~Q3) 내 포함 비율 ===")
for col, label, _ in metrics:
    for stratum in TARGET_STRATA_VIS:
        pop       = df_f[df_f['stratum'] == stratum][col].dropna()
        q1, q3    = pop.quantile(0.25), pop.quantile(0.75)
        samp      = df_target[df_target['stratum'] == stratum][col].dropna()
        in_iqr_n  = ((samp >= q1) & (samp <= q3)).sum()
        print(f"  {stratum} / {label}: {in_iqr_n}/{len(samp)} ({in_iqr_n/len(samp):.0%}) IQR 내")



=== 추출 표본 중 IQR(Q1~Q3) 내 포함 비율 ===
  large_high / 총 리뷰 수: 16/25 (64%) IQR 내
  mid_high / 총 리뷰 수: 12/23 (52%) IQR 내
  large_high / 소유자 수 하한: 22/25 (88%) IQR 내
  mid_high / 소유자 수 하한: 23/23 (100%) IQR 내
  large_high / 긍정 리뷰 비율: 11/25 (44%) IQR 내
  mid_high / 긍정 리뷰 비율: 11/23 (48%) IQR 내
